# Deep Learning Fundamentals - Mini-Project

## MNIST Digits - Scope and Architecture

## Project Scope & Objectives

This mini-project examines how hyperparameter choices shape learning dynamics and generalization of an **MLP** on MNIST. A **CNN** is included only as an optional comparison to contextualize results; both models are run through the same pipeline when compared.

Our main objectives are:

* Systematically vary key hyperparameters (batch size, learning rate, optimizer, weight decay, dropout, scheduler) and quantify their impact on loss, accuracy, and training dynamics.
* Capture metrics and artifacts consistently to enable correct and repeatable comparisons.
* Summarize findings with plots and tables that highlight sensitivity, trade-offs, and recommended settings.
* Optionally, contrast MLP and CNN under identical protocols to isolate architectural effects.

## Architecture & Components

We implemented our study on PyTorch Lightning, adding lightweight orchestration to keep experiments structured and reproducible. Our custom classes wrap training and evaluation while Lightning manages training loops.

This structure promoted the components decoupling and separation of concerns: the models focus on learning logic, the data module owns data lifecycle, and evaluators orchestrate experiments, therefore making it straightforward to compare configurations and reproduce results:

* **Models**: `MNISTMLP` (primary), `MNISTCNN` (optional) - subclasses of `LightningModule` implementing the core hooks for `fit`/`validate`/`test` (e.g., `training_step`, `validation_step`, `test_step`, `configure_optimizers`). The CNN analysis was only included as a dedicated comparison section.
* **Data**: `MNISTDataModule` - a `LightningDataModule` that encapsulates dataset download and preparation, transforms, and train/val/test loaders.
* **Evaluators**: `BatchSizeEvaluator`, `HyperparameterEvaluator` - utilities that construct a `Trainer`, attach a metrics callback, and execute controlled sweeps or batch-size studies with optional logging.

![Implementation Classes](images/classes.png "Implementation classes")

### Design Patterns

Besides the Separation of Concerns, the chosen architecture leveraged other design patterns which we found essential to enable our experiments flexibility and scalability:

- **Dependency Injection**: `HyperParameterEvaluator` receives a ready-to-use `MNISTDataModule` instance (`dm`) and a `model_cls` type, decoupling experiment orchestration from data/model construction. This allows swapping datasets or models without changing evaluator logic. 

- **Composition**: Both evaluators *compose* a `Trainer`, attach a lightweight inner `Callback` (`_MetricHistory`) to capture `val_acc`/`val_loss`, and optionally add a `CSVLogger`. This uses Lightning’s callback mechanism as an extension point while keeping evaluators cohesive and testable. `BatchSizeEvaluator` composes a fresh `MNISTDataModule` per batch size (strong ownership), whereas `HyperParameterEvaluator` reuses the injected `dm` (shared ownership).  

- **Template Method**: The models implement Lightning’s hook “template” (`training_step`, `validation_step`, `test_step`, `configure_optimizers`) while sharing common logic in `_shared_step`.

- **Strategy**: Optimizers, schedulers, and activations are chosen via parameters at runtime, mirroring a Strategy-like pattern effect by swapping behavior without touching the respective call sites.

## Code Quality & Conventions

We followed Python conventions and best practices throughout: clear type hints, docstrings when helpful, and explicit attributes that keep static analysis (Pylance) happy—avoiding blanket `# type: ignore` by making hyperparameters real fields and resolving doubtful references (e.g. using an explicit attribute binding in the model constructors).

We also encapsulated all training logic in importable classes (`MNISTMLP`, `MNISTCNN`, `MNISTDataModule`, evaluators), thus avoiding global methods and keeping notebooks focused on orchestration.

Finally, we emphasized reproducibility (fixed seeds), deterministic training where practical, and consistent logging/metrics that can capture and enable faithful comparisons.